# Case study, step 3: add the Facebook model to the neighbour tables

Adds the 100 nearest neighbours of each keyword in the off-the-shelf model cc.de.300 to the tables written by `05_analyse/05_casestudy_replication.ipynb` (`table_allneighbours_{Keyword}.csv`). Neighbours are lowercased, like for the self-trained models. If cc.de.300 is already in a table, that keyword is skipped, so the notebook can be re-run safely.

Requires `cc.de.300.bin` (from `00_download_facebook_model.ipynb`). Run from inside `06_casestudy/`. Next: `02_lemmatize_overlap.ipynb`.

In [ ]:
import os
import pandas as pd
import gensim

model_path = 'cc.de.300.bin'  # downloaded by 00_download_facebook_model.ipynb
model_name = 'cc.de.300'

keywords = {
    'frau':    'Frau',
    'femizid': 'Femizid',
    'mann':    'Mann',
    'mord':    'Mord',
    'täter':   'Täter',
    'opfer':   'Opfer',
}

# --- Load model once ---
print(f"Loading model: {model_name}")
model = gensim.models.fasttext.load_facebook_vectors(model_path)

# --- Append to each keyword's CSV ---
for keyword_lower, keyword_cased in keywords.items():
    csv_path = f'table_allneighbours_{keyword_cased}.csv'
    print(f"Processing keyword: '{keyword_cased}'")

    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
    else:
        print(f"  WARNING: {csv_path} not found — creating new file.")
        existing = pd.DataFrame(columns=['Model', 'Neighbor', 'Similarity'])

    if model_name in existing['Model'].values:
        print(f"  [{keyword_cased}] '{model_name}' already present — skipping.")
        continue

    try:
        neighbors = model.most_similar(keyword_cased, topn=100)
    except KeyError:
        print(f"  [{keyword_cased}] Keyword '{keyword_cased}' not in model vocab — skipping.")
        continue

    new_rows = pd.DataFrame([
        {'Model': model_name, 'Neighbor': neighbor.lower(), 'Similarity': similarity}
        for neighbor, similarity in neighbors
    ])

    updated = pd.concat([existing, new_rows], ignore_index=True)
    updated.to_csv(csv_path, index=False)
    print(f"  [{keyword_cased}] Added {len(new_rows)} neighbours → {csv_path}")

print("Done.")